# Matrixsteifigkeitsmethode – Tool

Dieses Notebook ist ein kompaktes FE-Tool für ebene Fachwerke.
Die gesamte Berechnungslogik ist in `fem_core.py` gekapselt; hier werden nur Modelldaten definiert und Ergebnisse ausgegeben.

In [ ]:
import numpy as np
from fem_core import assemble_K, solve_system
from fem_post import postprocessing, plot_results

## Input

Einheiten: Koordinaten und Längen in $[\text{mm}]$, Flächen in $[\text{mm}^2]$, E-Modul in $[\text{MPa}]$, Kräfte in $[\text{N}]$.

In [ ]:
# Knotenkoordinaten [X, Y] in mm
nodal_coordinates = np.array([
    [   0.0,    0.0],
    [1000.0,    0.0],
    [1000.0, 1000.0],
    [2000.0, 1000.0],
])

# Elemente: [Knoten_i, Knoten_j, section_key]
elements = [
    [0, 1, "section 1"],
    [0, 2, "section 2"],
    [1, 2, "section 3"],
    [1, 3, "section 4"],
    [2, 3, "section 5"],
]

# Materialien: E-Modul [MPa = N/mm²]
materials = {"steel": [210000.0]}

# Querschnitte: [A [mm²], material_key]
sections = {
    "section 1": [15.00, "steel"],
    "section 2": [28.28, "steel"],
    "section 3": [10.00, "steel"],
    "section 4": [56.56, "steel"],
    "section 5": [10.00, "steel"],
}

# Randbedingungen: [Knoten (0-basiert), Achse (0=x,1=y), vorgegebene Verschiebung]
constraints = [
    [0, 0, 0.0],   # Knoten 1: U_1 = 0 (Festlager, x)
    [0, 1, 0.0],   # Knoten 1: U_2 = 0 (Festlager, y)
    [1, 1, 0.0],   # Knoten 2: U_4 = 0 (Loslager, y)
]

# Lasten: [Knoten (0-basiert), Achse (0=x,1=y), Kraft [N]]
loads = [
    [3, 1, -1000.0],   # Knoten 4: F_y = -1000 N
]

## Core

In [ ]:
K            = assemble_K(nodal_coordinates, elements, sections, materials)
U, F, fixed  = solve_system(K, constraints, loads)
eps, sig, N  = postprocessing(U, nodal_coordinates, elements, sections, materials)

## Ergebnisse

In [ ]:
print("Verschiebungen und Kräfte:")
print(f"  {'DOF':>3}  {'U [mm]':>16}  {'F [N]':>12}")
print("  " + "─" * 36)
for k in range(len(U)):
    print(f"  {k+1:>3}  {U[k]:>+16.6e}  {F[k]:>+12.4f}")

print("\nStabkräfte:")
print(f"  {'Stab':>4}  {'ε [-]':>14}  {'σ [MPa]':>10}  {'N [N]':>10}")
print("  " + "─" * 46)
for e in range(len(elements)):
    print(f"  {e+1:>4}  {eps[e]:>14.6e}  {sig[e]:>10.4f}  {N[e]:>10.4f}")

## Visualisierung

In [ ]:
plot_results(nodal_coordinates, elements, constraints, loads, U, sig, scale=200)